# 3.1 Probability Fundamentals

**Goal:** define experiments, outcomes, sample spaces, and events before assigning probabilities.

**Prerequisites:** Module 0 set notation, fractions, and percentages; Module 1 populations and samples. **Runtime:** Python 3.10+, standard library only. All examples are synthetic or idealized models. Saved outputs are included; use **Restart Kernel and Run All** to reproduce them.

## Contents
1. What is probability?
2. Random experiments
3. Outcomes
4. Sample spaces
5. Events
6. Equally likely events and outcomes
7. Complementary events
8. Simulation and ML interpretation
9. Practice and solutions

## 1. What is Probability?

Probability is a mathematical measure of uncertainty. For an event $A$, $P(A)$ is a number between 0 and 1: 0 represents an impossible event in a finite positive-mass model, 1 a certain event, and intermediate values quantify uncertainty under the stated model or information.

A probability model consists of a sample space, a collection of events, and a probability assignment. Its basic axioms are:

1. $P(A)\ge0$ for every event.
2. $P(\Omega)=1$ for the full sample space $\Omega$.
3. For countably many pairwise disjoint events, $P(\bigcup_i A_i)=\sum_iP(A_i)$.

These imply $P(\varnothing)=0$ and $P(A)\le1$. Probability statements depend on a model and available information: “a fair die has probability 1/6 of landing on 4” uses assumptions about the die and roll, not a conclusion from notation alone.

A **frequentist** interpretation relates probability to long-run behavior under repetition. A **Bayesian** interpretation uses probability to quantify uncertainty given information, including uncertainty about hypotheses or parameters. Both use the probability rules in these notebooks. An observed relative frequency estimates a probability in suitable circumstances; it need not equal it exactly.

For continuous models, an exact point can have probability zero while remaining a possible value, and probability-one events need not be logically unavoidable. The finite models here avoid that subtlety; Module 4 develops continuous distributions.

## 2. Random Experiments

A random experiment is a procedure with well-defined possible results whose realized result is uncertain before observation. Examples include rolling a die, drawing a card, observing whether a request fails, or selecting a user through a random sampling design.

Specify the procedure: one draw or two; with or without replacement; fair or biased; ordered or unordered; independent repetitions or dependent observations. These choices change the model.

**Worked example:** draw a colored ball from a bag with 3 red and 1 blue ball. If each physical ball is equally likely to be drawn, red has probability 3/4. The two color labels are not automatically equally probable.

In [1]:
from fractions import Fraction
physical_balls = ["R1", "R2", "R3", "B1"]
red_balls = [ball for ball in physical_balls if ball.startswith("R")]
print("Physical outcomes:", physical_balls)
print("P(red):", Fraction(len(red_balls), len(physical_balls)))
assert Fraction(len(red_balls), len(physical_balls)) == Fraction(3, 4)

Physical outcomes: ['R1', 'R2', 'R3', 'B1']
P(red): 3/4


## 3. Outcomes

An **outcome** is one elementary result at the chosen resolution. A die outcome can be 4; a two-coin outcome can be the ordered pair (H,T). An **event** may contain several outcomes; do not confuse “the die is even” with a single face.

Resolution matters. Recording only the number of heads in two tosses produces outcomes 0,1,2; recording the full ordered sequence produces HH,HT,TH,TT. These are different sample spaces describing the same experiment at different levels of detail, with different probability assignments.

## 4. Sample Spaces

The sample space $\Omega$ contains all possible elementary outcomes for the modeled experiment. It must cover every result relevant to the stated procedure, and outcomes must be distinct at the chosen resolution.

**Worked example:** two independent fair coin tosses have ordered sample space
$$\Omega=\{HH,HT,TH,TT\}.$$
Each sequence has probability 1/4. The number of heads instead has probabilities $P(0)=1/4$, $P(1)=1/2$, $P(2)=1/4$. Counting three labels and assigning 1/3 each would be wrong.

Sample spaces can be finite, countably infinite (the toss number of the first head), or continuous (a modeled response time). The favorable-over-total counting rule applies only to finite equally likely elementary outcomes, not automatically to every space.

In [2]:
from itertools import product
from collections import Counter
sequences = list(product("HT", repeat=2))
head_counts = Counter(sequence.count("H") for sequence in sequences)
print("Ordered sequences:", sequences)
print("Head-count probabilities:", {k: Fraction(v, len(sequences)) for k, v in sorted(head_counts.items())})
assert head_counts[1] == 2

Ordered sequences: [('H', 'H'), ('H', 'T'), ('T', 'H'), ('T', 'T')]
Head-count probabilities: {0: Fraction(1, 4), 1: Fraction(1, 2), 2: Fraction(1, 4)}


## 5. Events

An event is a subset of the sample space. For a die, $A=\{2,4,6\}$ means “even,” and $B=\{4,5,6\}$ means “greater than 3.” An event occurs if the observed outcome belongs to its set.

- A **simple/elementary event** contains one outcome.
- A **compound event** contains multiple outcomes.
- The certain event is $\Omega$; the impossible event is $\varnothing$.

For a finite model with elementary probabilities $p(\omega)$,
$$P(A)=\sum_{\omega\in A}p(\omega),\qquad\sum_{\omega\in\Omega}p(\omega)=1.$$
The fair-die formula $P(A)=|A|/|\Omega|$ is the special case with equal elementary masses. If the die is biased, use the actual masses instead.

In [3]:
omega = set(range(1, 7))
even = {2, 4, 6}
biased_pmf = {1: Fraction(1, 10), 2: Fraction(1, 10), 3: Fraction(1, 10),
              4: Fraction(1, 10), 5: Fraction(1, 10), 6: Fraction(1, 2)}
assert sum(biased_pmf.values()) == 1
print("Fair P(even):", Fraction(len(even), len(omega)))
print("Biased P(even):", sum(biased_pmf[x] for x in even))
assert sum(biased_pmf[x] for x in even) == Fraction(7, 10)

Fair P(even): 1/2
Biased P(even): 7/10


## 6. Equally Likely Events

Events $A$ and $B$ are equally likely when $P(A)=P(B)$. They need not be the same event or be independent. On a fair die, “even” and “odd” each have probability 1/2; they are also mutually exclusive.

**Equally likely elementary outcomes** justify
$$P(A)=\frac{\text{number of favorable elementary outcomes}}{\text{total number of elementary outcomes}}.$$

**Worked example:** the two dice are distinguishable and independently fair, so the 36 ordered pairs are equally likely. The event “sum is 7” has six pairs, giving $6/36=1/6$. The possible sums 2 through 12 are not equally likely because they correspond to different numbers of pairs.

A practical checklist before counting: define the elementary outcomes, justify equal likelihood, and count each outcome once. Apparent symmetry of labels is not sufficient evidence of fairness.

In [4]:
dice_pairs = list(product(range(1, 7), repeat=2))
sum_seven = [pair for pair in dice_pairs if sum(pair) == 7]
print("Sum-seven pairs:", sum_seven)
print("P(sum=7):", Fraction(len(sum_seven), len(dice_pairs)))
print("P(sum=2):", Fraction(sum(sum(pair) == 2 for pair in dice_pairs), len(dice_pairs)))
assert Fraction(len(sum_seven), len(dice_pairs)) == Fraction(1, 6)

Sum-seven pairs: [(1, 6), (2, 5), (3, 4), (4, 3), (5, 2), (6, 1)]
P(sum=7): 1/6
P(sum=2): 1/36


## 7. Complementary Events

The complement $A^c=\Omega\setminus A$ means “not A.” Because $A$ and $A^c$ are disjoint and cover the whole space,
$$P(A^c)=1-P(A).$$

**Worked example:** with two independent fair dice, “at least one six” is the complement of “no six.” Its probability is $1-(5/6)^2=11/36$. “Not at least one” means none; it does not mean “exactly one.” Exactly one six instead has probability $2(1/6)(5/6)=10/36$.

Complement methods are particularly useful for “at least one” questions when “none” is simpler to calculate. Independence must still be justified before multiplying the no-event probabilities.

In [5]:
at_least_one = [pair for pair in dice_pairs if 6 in pair]
exactly_one = [pair for pair in dice_pairs if pair.count(6) == 1]
via_complement = 1 - Fraction(5, 6)**2
print("At least one six:", via_complement)
print("Exactly one six:", Fraction(len(exactly_one), len(dice_pairs)))
assert via_complement == Fraction(len(at_least_one), len(dice_pairs)) == Fraction(11, 36)

At least one six: 11/36
Exactly one six: 5/18


## 8. Simulation and AI/ML interpretation

Simulation generates outcomes from a specified model. Empirical proportions can illustrate probability rules, but simulation does not validate the model's realism. Under independent repetitions, larger samples generally reduce typical random error; estimates need not get closer to truth at every step.

In ML, a predicted probability 0.8 is a model's numerical assessment, not a guarantee that this one case will have outcome 1. **Calibration** asks whether cases receiving similar probabilities have corresponding observed frequencies, using an appropriate evaluation dataset. Uncertainty can reflect outcome randomness, limited evidence, model assumptions, and distribution mismatch.

In [6]:
import random
rng = random.Random(31)
rolls = [(rng.randint(1, 6), rng.randint(1, 6)) for _ in range(20000)]
for size in [100, 1000, 20000]:
    estimate = sum(6 in pair for pair in rolls[:size])/size
    print(f"n={size}: estimated P(at least one six)={estimate:.4f}")
print("Exact model probability:", float(Fraction(11, 36)))

n=100: estimated P(at least one six)=0.3000
n=1000: estimated P(at least one six)=0.3210
n=20000: estimated P(at least one six)=0.3103
Exact model probability: 0.3055555555555556


## Practice

1. State the experiment, ordered sample space, and event “exactly one head” for two coin tosses.
2. A fair die is rolled. Find the probabilities of a prime number and its complement.
3. Why is the sum of two fair dice not uniform over 2 through 12?
4. A bag contains 2 red and 3 blue balls. Under uniform drawing of physical balls, find $P(\text{blue})$.
5. Find the probability of at least one head in three independent fair tosses.
6. Can equally likely events be dependent? Use even and odd outcomes on a fair die.
7. Does a probability estimate of 0.9 guarantee that the next outcome is positive?

## Solutions

1. Experiment: toss twice and record order. Space HH,HT,TH,TT; event {HT,TH}, probability 1/2 under independence and fairness.
2. Primes {2,3,5}, probability 3/6=1/2; complement {1,4,6}, also 1/2. The number 1 is not prime.
3. Sums have different numbers of ordered pairs: one pair for 2, six for 7. It is the 36 pairs that are equally likely.
4. 3/5, not 1/2 merely because there are two color labels.
5. Complement of all tails: $1-(1/2)^3=7/8$.
6. Yes. They have equal probability 1/2 but cannot occur together, so their intersection 0 differs from the product 1/4.
7. No. It expresses uncertainty under information/model assumptions. Reliability requires assessment over relevant repeated cases.

**Readiness check:** define the space before counting, separate an outcome from an event, and justify both equal likelihood and independence where used.

**Next:** [3.2 Probability Rules](2_Probability-Rules.ipynb).